# Lab 42 — Computación Cuántica Distribuida y Redes Cuánticas

Este laboratorio explora las técnicas fundamentales de la computación cuántica distribuida (DQC): entanglement swapping, delegated quantum computation (DQC), y criptografía cuántica avanzada.

**Contenido:**
- Parte A: Entanglement Swapping y repetidores cuánticos
- Parte B: Blind Quantum Computation (BQC) — delegar cómputos preservando privacidad
- Parte C: Distribución de claves cuánticas (QKD) avanzada — BB84 vs E91
- Parte D: Telepuerta cuántica no local y LOCC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector, partial_trace, entropy, DensityMatrix
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

print('Qiskit y Aer listos para DQC')

## Parte A: Entanglement Swapping

El *entanglement swapping* es el mecanismo que permite a dos nodos que **nunca han interactuado** quedar entrelazados.

**Protocolo:**
1. Alice-Bob comparten Bell: |Φ⁺⟩_{AB}
2. Charlie-Dave comparten Bell: |Φ⁺⟩_{CD}
3. Bob y Charlie aplican medición de Bell sobre sus qubits
4. Alice y Dave quedan entrelazados (condicionalmente)

**Estado inicial total:**
$$|\Psi\rangle = |\Phi^+\rangle_{AB} \otimes |\Phi^+\rangle_{CD} = \frac{1}{2}(|0000\rangle + |0011\rangle + |1100\rangle + |1111\rangle)_{ABCD}$$

In [ ]:
def entanglement_swapping_circuit() -> QuantumCircuit:
    """Circuito de entanglement swapping: 4 qubits ABCD, medición en BC."""
    qr = QuantumRegister(4, 'q')  # q0=A, q1=B, q2=C, q3=D
    cr = ClassicalRegister(2, 'c')  # medición de B y C
    qc = QuantumCircuit(qr, cr)

    # Crear Bell AB
    qc.h(0)
    qc.cx(0, 1)

    # Crear Bell CD
    qc.h(2)
    qc.cx(2, 3)

    qc.barrier()

    # Medición de Bell en BC (operación de swap)
    qc.cx(1, 2)
    qc.h(1)
    qc.measure(1, 0)  # mide B
    qc.measure(2, 1)  # mide C

    # Correcciones condicionales en AD
    with qc.if_test((cr[1], 1)):
        qc.x(3)
    with qc.if_test((cr[0], 1)):
        qc.z(3)

    return qc


qc_swap = entanglement_swapping_circuit()
qc_swap.draw('mpl', fold=-1)

In [ ]:
# Verificar que A y D quedan entrelazados después del swap
# Usando Statevector sin las mediciones para analizar el estado
qc_verify = QuantumCircuit(4)
# Crear Bell AB
qc_verify.h(0)
qc_verify.cx(0, 1)
# Crear Bell CD
qc_verify.h(2)
qc_verify.cx(2, 3)
# Aplicar operación de Bell en BC
qc_verify.cx(1, 2)
qc_verify.h(1)

sv = Statevector(qc_verify)
print('Estado |ABCD⟩ antes de medir BC:')
sv_dict = sv.to_dict()
for state, amp in sorted(sv_dict.items()):
    if abs(amp) > 0.01:
        print(f'  |{state}⟩: {amp:.4f}')

# Calcular entropía de entrelazamiento de A con BCD (qubit 0 = A en Qiskit es el último)
rho_full = DensityMatrix(sv)
rho_A = partial_trace(rho_full, [1, 2, 3])  # traza sobre BCD
print(f'\nMatriz densidad de A: Tr(ρ_A²) = {np.trace(rho_A.data @ rho_A.data).real:.4f}')

# Calcular entropía para el par AD (qubits 0 y 3)
rho_BC = partial_trace(rho_full, [0, 3])  # traza sobre A y D → estado de BC
print(f'Entropía de entrelazamiento S(B) = {entropy(partial_trace(rho_full, [0, 2, 3])):.4f}')

## Parte B: Blind Quantum Computation (BQC)

En BQC (protocolo de Broadbent-Fitzsimons-Kashefi), un cliente con capacidades limitadas delega un cómputo a un servidor sin revelarle el circuito.

**Ingredientes:**
- Cliente prepara qubits rotados: |+_θ⟩ = (|0⟩ + e^{iθ}|1⟩)/√2
- Servidor aplica el circuito en MBQC (measurement-based QC) sobre un cluster state
- Ángulos de medición del cliente están enmascarados: δ = φ + θ + rπ
- Servidor no puede distinguir el cómputo real del ruido intencional

In [ ]:
def prepare_rotated_qubit(theta: float) -> QuantumCircuit:
    """Prepara |+_θ⟩ = (|0⟩ + exp(iθ)|1⟩)/√2."""
    qc = QuantumCircuit(1)
    qc.h(0)
    qc.p(theta, 0)
    return qc


def bqc_single_qubit_gate(phi: float, r: int, theta: float) -> float:
    """Calcula ángulo de medición enmascarado para BQC.

    phi: ángulo de computación deseado
    r: bit de randomización {0, 1}
    theta: ángulo secreto del cliente
    """
    return phi + theta + r * np.pi


# Demostrar que el servidor no puede inferir phi de delta
phi_secret = np.pi / 4  # ángulo que el cliente quiere aplicar
n_trials = 1000

deltas = []
for _ in range(n_trials):
    theta = np.random.uniform(0, 2 * np.pi)  # theta uniforme en [0, 2π]
    r = np.random.randint(0, 2)              # bit aleatorio
    delta = bqc_single_qubit_gate(phi_secret, r, theta) % (2 * np.pi)
    deltas.append(delta)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(deltas, bins=50, color='steelblue', alpha=0.7, density=True)
ax.axhline(1 / (2 * np.pi), color='red', linestyle='--', label='Distribución uniforme')
ax.set_xlabel('δ (ángulo de medición observado por el servidor)')
ax.set_ylabel('Densidad')
ax.set_title('BQC: El servidor ve δ uniformemente distribuido — φ es indistinguible')
ax.legend()
plt.tight_layout()
plt.show()

print(f'φ secreto: {phi_secret:.4f} rad = {np.degrees(phi_secret):.1f}°')
print(f'Media de δ (debería ser uniforme ≈ π): {np.mean(deltas):.4f}')
print(f'Varianza de δ (debería ser π²/3 ≈ {np.pi**2/3:.3f}): {np.var(deltas):.4f}')

## Parte C: QKD — Comparativa BB84 vs E91

### BB84 (Bennett-Brassard 1984)
- Alice envía qubits en 4 estados: {|0⟩, |1⟩, |+⟩, |−⟩}
- Bob mide en base Z o X aleatoriamente
- Reconciliación pública de bases → ~50% de bits de clave
- Seguridad: Eve introduce QBER ≥ 25% al interceptar

### E91 (Ekert 1991)
- Alice y Bob comparten pares entrelazados |Φ⁺⟩
- Miden en 3 bases cada uno → correlaciones de Bell
- Seguridad: violación CHSH S > 2 garantiza ausencia de espía

In [ ]:
def simulate_bb84(n_bits: int, eve_present: bool = False, p_eve: float = 0.5) -> dict:
    """Simula BB84 con o sin espía Eve.

    Retorna tasa de clave, QBER y bits de clave compartidos.
    """
    np.random.seed(42)
    # Alice
    alice_bits = np.random.randint(0, 2, n_bits)    # bits a enviar
    alice_bases = np.random.randint(0, 2, n_bits)   # 0=Z, 1=X
    # Bob
    bob_bases = np.random.randint(0, 2, n_bits)

    # Eve (si presente, mide y reenvía en base aleatoria)
    if eve_present:
        eve_bases = np.random.randint(0, 2, n_bits)
        # Eve solo intercepta con probabilidad p_eve
        intercept_mask = np.random.random(n_bits) < p_eve
        # Eve introduce errores cuando usa base diferente a Alice
        eve_errors = (eve_bases != alice_bases) & intercept_mask
    else:
        eve_errors = np.zeros(n_bits, dtype=bool)

    # Coincidencia de bases → bits de clave
    matching = alice_bases == bob_bases
    raw_key_alice = alice_bits[matching]

    # Bits de Bob: correctos si bases coinciden y Eve no introdujo error
    bob_errors = eve_errors[matching]
    raw_key_bob = raw_key_alice.copy()
    raw_key_bob[bob_errors] ^= 1  # flip por interferencia de Eve

    qber = np.mean(raw_key_alice != raw_key_bob) if len(raw_key_alice) > 0 else 0
    key_rate = len(raw_key_alice) / n_bits

    return {
        'n_bits': n_bits,
        'key_length': len(raw_key_alice),
        'key_rate': key_rate,
        'qber': qber,
        'secure': qber < 0.11,  # umbral QBER≈11% para BB84
    }


# Comparar escenarios
scenarios = [
    ('Sin Eve', simulate_bb84(1000, eve_present=False)),
    ('Eve 25%', simulate_bb84(1000, eve_present=True, p_eve=0.25)),
    ('Eve 50%', simulate_bb84(1000, eve_present=True, p_eve=0.50)),
    ('Eve 100%', simulate_bb84(1000, eve_present=True, p_eve=1.00)),
]

print(f"{'Escenario':<12} {'Bits enviados':>14} {'Clave raw':>10} {'QBER':>8} {'Seguro':>8}")
print('-' * 58)
for name, res in scenarios:
    secure = '✅' if res['secure'] else '❌'
    print(f"{name:<12} {res['n_bits']:>14} {res['key_length']:>10} {res['qber']:.3f} {secure:>8}")

In [ ]:
# Curva QBER vs fracción de qubits interceptados por Eve
intercept_fracs = np.linspace(0, 1, 50)
qbers = [simulate_bb84(2000, eve_present=True, p_eve=f)['qber'] for f in intercept_fracs]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(intercept_fracs * 100, np.array(qbers) * 100, 'b-', lw=2, label='QBER BB84 simulado')
ax.axhline(25, color='orange', ls='--', label='Máx. QBER teórico Eve (25%)')
ax.axhline(11, color='red', ls='-.', label='Umbral seguridad QBER=11%')
ax.fill_between(intercept_fracs * 100, 0, 11, alpha=0.15, color='green', label='Zona segura')
ax.fill_between(intercept_fracs * 100, 11, 30, alpha=0.15, color='red', label='Zona insegura')
ax.set_xlabel('% bits interceptados por Eve')
ax.set_ylabel('QBER (%)')
ax.set_title('BB84: QBER en función de la interceptación de Eve')
ax.legend()
ax.set_xlim(0, 100)
ax.set_ylim(0, 30)
plt.tight_layout()
plt.show()

print('\nEn BB84, Eve interceptando el 100% introduce QBER ≈ 25%')
print('La presencia de Eve es detectable si QBER supera el umbral ≈ 11%')

## Parte D: Telepuerta no-local y LOCC

Las operaciones LOCC (Local Operations and Classical Communication) son las operaciones "libres" en la teoría del entrelazamiento. La teletransportación es el uso del entrelazamiento como recurso.

**Capacidad del canal cuántico:** dado un canal ruidoso ε, la capacidad de entrelazamiento es:
$$E_D(\rho) = \lim_{n\to\infty} \frac{1}{n} \max_{\text{LOCC}} E(\Lambda_{\text{LOCC}}(\rho^{\otimes n}))$$

In [ ]:
def teleportation_fidelity(p_depol: float, n_shots: int = 2048) -> float:
    """Calcula la fidelidad de teletransportación con ruido depolarizante en el canal."""
    # Estado a teleportar: |+⟩ = H|0⟩
    target_sv = Statevector.from_label('0').evolve(QuantumCircuit(1).h(0) if False else _h_gate())
    # Simplificado: comparar salida con entrada usando simulador ruidoso
    fidelities = []

    # Construir circuito de teletransportación
    qc = QuantumCircuit(3, 2)  # q0=estado, q1=Alice, q2=Bob
    qc.h(0)  # preparar |+⟩
    qc.h(1)  # Bell entre Alice y Bob
    qc.cx(1, 2)
    qc.cx(0, 1)
    qc.h(0)
    qc.measure(0, 0)
    qc.measure(1, 1)
    with qc.if_test((qc.clbits[1], 1)):
        qc.x(2)
    with qc.if_test((qc.clbits[0], 1)):
        qc.z(2)

    # Ruido depolarizante en el qubit de canal
    noise_model = NoiseModel()
    if p_depol > 0:
        error = depolarizing_error(p_depol, 1)
        noise_model.add_quantum_error(error, ['h', 'cx'], [1])  # ruido en qubit canal
        noise_model.add_quantum_error(error, ['h', 'cx'], [2])

    sim = AerSimulator(noise_model=noise_model)
    job = sim.run(qc, shots=n_shots)
    counts = job.result().get_counts()

    # Fidelidad aproximada: sin ruido todos los resultados dan |+⟩ correcto
    # Con ruido depolarizante, F = 1 - 2p/3 para 1-qubit
    return max(0.0, 1.0 - 2 * p_depol / 3.0)


def _h_gate():
    qc = QuantumCircuit(1)
    qc.h(0)
    return qc


# Curva de fidelidad vs ruido en el canal de Bell
p_vals = np.linspace(0, 0.5, 30)
fidelities_teleport = [teleportation_fidelity(p) for p in p_vals]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(p_vals, fidelities_teleport, 'b-o', ms=4, label='F teletransportación (depolarizante)')
ax.axhline(2/3, color='red', ls='--', label='Límite clásico F_cl = 2/3')
ax.axhline(1.0, color='green', ls=':', label='Fidelidad perfecta = 1')
ax.fill_between(p_vals, 2/3, fidelities_teleport, where=np.array(fidelities_teleport) > 2/3,
                alpha=0.2, color='blue', label='Ventaja cuántica')
ax.set_xlabel('Tasa de error depolarizante p')
ax.set_ylabel('Fidelidad de teletransportación')
ax.set_title('Teletransportación vs canal ruidoso — ventaja sobre protocolo clásico')
ax.legend()
ax.set_ylim(0.4, 1.05)
plt.tight_layout()
plt.show()

p_crossover = 1/8  # F = 1 - 2p/3 = 2/3  →  p = 1/6 ≈ 0.167
print(f'La teletransportación supera el límite clásico para p < {1/6:.3f} ≈ 16.7%')

## Resumen

| Técnica | Recurso cuántico | Comunicación clásica | Aplicación |
|---------|-----------------|---------------------|------------|
| Entanglement swapping | 2 pares Bell | 2 bits | Repetidores cuánticos |
| BQC | Estado de cluster | Ángulos enmascarados | Cloud cuántico privado |
| BB84 | Qubits individuales | Reconciliación de bases | QKD punto a punto |
| E91 | Pares entrelazados | Resultados de medición | QKD con prueba Bell |
| Teletransportación | 1 par Bell | 2 bits clásicos | Transferencia de estado |

**Límites fundamentales:**
- La no-clonación impide amplificar estados cuánticos desconocidos
- La señalización más rápida que la luz está prohibida (no-communication theorem)
- La destilación de entrelazamiento LOCC requiere muchas copias del estado